# TP53 Mutation Analysis — full pipeline on Kaggle
Reproducible ML + structural analysis of disease-associated TP53 variants, with a **3-model side-by-side benchmark**.

**Setup:** turn **Internet ON** (right-side Settings) for the repo clone and live ClinVar fetch. No GPU needed. Runtime ~10–15 min. Every result below is computed live in this notebook.

## 1 · Code and dependencies

In [ ]:
!git clone https://github.com/simbisaichinhema/TP53-Sequence-Structure-and-Machine-Learning-.git 2>&1 | tail -1
%cd TP53-Sequence-Structure-and-Machine-Learning-
!pip install -q pandas scikit-learn matplotlib joblib xgboost lightgbm catboost
import sklearn, pandas, matplotlib
print('sklearn', sklearn.__version__, '| pandas', pandas.__version__, '| matplotlib', matplotlib.__version__)
for m in ('xgboost','lightgbm','catboost'):
    try:
        mod=__import__(m); print(m, getattr(mod,'__version__','?'))
    except Exception as e:
        print(m, 'MISSING — benchmark will skip that model')

## 2 · WT baseline (verified reference — reused, not rebuilt)

In [ ]:
!python3 scripts/sequence/verify_sequence.py --reference data/raw/tp53_cds/P04637_uniprot_live_20260918.fasta 2>&1 | tail -6
!python3 scripts/structure/superpose_core.py 2>&1 | tail -4

## 3 · Variant dataset — ClinVar clear labels + conflicting majority-vote expansion

Target size **500–1000**. Clear germline P/LP and B/LB labels are kept as-is; ClinVar *conflicting* missense records are resolved by a **strict majority** of submitter `GermlineClassification` values (ties / VUS majorities dropped). VUS are never given invented labels.

In [ ]:
import os
import pandas as pd
path = 'data/processed/mutations/tp53_variant_dataset.csv'
need_expand = True
if os.path.exists(path):
    _df0 = pd.read_csv(path)
    if len(_df0) >= 500:
        need_expand = False
        print(f'Using existing expanded dataset n={len(_df0)}')
if need_expand:
    if not os.path.exists('data/raw/variants/clinvar_tp53_esummary.json'):
        !python3 scripts/mutation/fetch_clinvar_tp53.py
    !python3 scripts/mutation/expand_clinvar_conflicts.py --sleep 0.35
df = pd.read_csv(path)
print(df['label_basis'].value_counts().to_string())
assert 500 <= len(df) <= 1200, f'expected 500–1000+ variants, got {len(df)}'

## 4 · Dataset summary (real counts)

In [ ]:
df = pd.read_csv('data/processed/mutations/tp53_variant_dataset.csv')
print(f"n={len(df)} | pathogenic={int((df.label==1).sum())} | benign={int((df.label==0).sum())} | "
      f"dupes={df.duplicated(['pos','ref','alt']).sum()} | "
      f"positions {df.pos.min()}–{df.pos.max()} | unique positions {df.pos.nunique()}")
print('label basis:')
print(df.label_basis.value_counts().to_string())
df[df.hgvs_p.isin(['p.R175H','p.R248Q','p.R248W','p.R273H','p.G245S','p.Y220C','p.P72R'])][['hgvs_p','classification','review_status','label_basis']]

## 5 · Feature extraction — 21 leak-free sequence/structural features

In [ ]:
!python3 scripts/machine_learning/make_features.py
feat = pd.read_csv('data/processed/structural_features/tp53_features.csv')
print(feat.shape)
feat[['hgvs_p','pos','ref','alt','label','grantham','plddt_at_pos','in_core']].head(5)

## 6 · Baseline train (logistic regression + random forest)
Stratified 80/20, seed 42, scaler inside pipeline, 5-fold CV on train only.

In [ ]:
!python3 scripts/machine_learning/train.py

## 7 · Multi-model benchmark — 3 algorithms, fair protocol

Same feature matrix · same stratified 80/20 split (seed 42) · same 5-fold CV on train · scaling only where needed · test set never used for selection.

Models (default set, top-3 by train CV ROC-AUC): Extra Trees, Random Forest, CatBoost. Others remain available via `--models`.

In [ ]:
!python3 scripts/machine_learning/benchmark_models.py

## 8 · Side-by-side model comparison (held-out test, n = 20%)

All metrics below are measured on the **same** held-out test set. CV columns are mean ± SD over 5 folds on the training partition only.

In [ ]:
import pandas as pd
from IPython.display import display

bench = pd.read_csv('results/machine_learning/benchmark/benchmark_results.csv')

# --- Table A: full test-set metrics, sorted by ROC-AUC ---
test_cols = ['model','test_accuracy','test_balanced_accuracy','test_precision',
             'test_recall','test_specificity','test_f1','test_roc_auc',
             'test_pr_auc','test_mcc','test_tn','test_fp','test_fn','test_tp']
test_tbl = (bench[test_cols]
            .rename(columns={c: c.replace('test_','') for c in test_cols if c!='model'})
            .sort_values('roc_auc', ascending=False)
            .reset_index(drop=True))
test_tbl.insert(0, 'rank', range(1, len(test_tbl)+1))
print('=== TABLE A — Held-out test metrics (same split for every model) ===')
display(test_tbl.style.format({
    c: '{:.3f}' for c in ['accuracy','balanced_accuracy','precision','recall',
                          'specificity','f1','roc_auc','pr_auc','mcc']
}).hide(axis='index'))

# --- Table B: CV mean ± SD vs test, baseline vs challengers ---
BASELINES = {'logistic_regression','random_forest'}
def cv_pm(row, metric):
    return f"{row[f'cv_{metric}_mean']:.3f} ± {row[f'cv_{metric}_std']:.3f}"

cmp_rows = []
for _, r in bench.iterrows():
    cmp_rows.append({
        'model': r['model'],
        'role': 'baseline' if r['model'] in BASELINES else 'challenger',
        'CV accuracy': cv_pm(r, 'accuracy'),
        'CV F1': cv_pm(r, 'f1'),
        'CV ROC-AUC': cv_pm(r, 'roc_auc'),
        'CV PR-AUC': cv_pm(r, 'pr_auc'),
        'CV MCC': cv_pm(r, 'mcc'),
        'test acc': round(r['test_accuracy'], 3),
        'test F1': round(r['test_f1'], 3),
        'test ROC-AUC': round(r['test_roc_auc'], 3),
        'test PR-AUC': round(r['test_pr_auc'], 3),
        'test MCC': round(r['test_mcc'], 3),
    })
cmp = pd.DataFrame(cmp_rows).sort_values('test ROC-AUC', ascending=False).reset_index(drop=True)
print('=== TABLE B — Baselines vs challengers (CV mean±SD | test) ===')
display(cmp.style.hide(axis='index'))

# --- Head-to-head deltas vs each baseline ---
print('=== TABLE C — Δ test ROC-AUC / Δ test F1 vs baselines (positive = better) ===')
for base in ['logistic_regression','random_forest']:
    br = bench[bench.model==base].iloc[0]
    d = bench[['model','test_roc_auc','test_f1']].copy()
    d['d_auc'] = (d['test_roc_auc'] - br['test_roc_auc']).round(4)
    d['d_f1'] = (d['test_f1'] - br['test_f1']).round(4)
    d = d.sort_values('d_auc', ascending=False).reset_index(drop=True)
    print(f'\nvs {base}  (AUC={br.test_roc_auc:.3f}, F1={br.test_f1:.3f})')
    display(d.rename(columns={
        'test_roc_auc':'AUC','test_f1':'F1','d_auc':'ΔAUC','d_f1':'ΔF1'
    }).style.hide(axis='index'))

best = bench.sort_values('test_roc_auc', ascending=False).iloc[0]
print(f"Best test ROC-AUC: {best.model} = {best.test_roc_auc:.3f} "
      f"(F1={best.test_f1:.3f}, MCC={best.test_mcc:.3f})")
print('Selection uses CV ROC-AUC on train only — see benchmark_report.md.')

## 9 · Side-by-side comparison figures

Accuracy · F1 · ROC-AUC · PR-AUC bars (CV mean ± SD), combined ROC curves, precision–recall curves, confusion-matrix grid, CV variability.

In [ ]:
from IPython.display import Image, display
figs = [
    'figures/machine_learning/benchmark/accuracy_comparison.png',
    'figures/machine_learning/benchmark/f1_comparison.png',
    'figures/machine_learning/benchmark/roc_auc_comparison.png',
    'figures/machine_learning/benchmark/pr_auc_comparison.png',
    'figures/machine_learning/benchmark/roc_curves.png',
    'figures/machine_learning/benchmark/precision_recall_curves.png',
    'figures/machine_learning/benchmark/confusion_matrices.png',
    'figures/machine_learning/benchmark/cv_performance_variability.png',
]
for f in figs:
    print(f)
    display(Image(f, width=680))

## 10 · Baseline metrics + classic figures (train.py)

In [ ]:
import json
m = json.load(open('results/machine_learning/metrics.json'))
rows = [{'model': n, 'CV-F1': f"{v['cv_f1_mean']}±{v['cv_f1_std']}", 'acc': v['accuracy'],
         'prec': v['precision'], 'rec': v['recall'], 'F1': v['f1'], 'ROC-AUC': v['roc_auc']}
        for n, v in m['models'].items()]
display(pd.DataFrame(rows).set_index('model'))
print('selected:', m['selected_model'], '| top features:', ', '.join(m['feature_importance_ranking'][:5]))
from IPython.display import Image
for f in ['figures/machine_learning/confusion_matrix.png', 'figures/machine_learning/roc_curve.png',
          'figures/machine_learning/feature_importance.png', 'figures/machine_learning/class_distribution.png']:
    display(Image(f, width=620))

## 11 · Hyperparameter tuning (train-only, top-3 by CV ROC-AUC)
Baseline results stay in `benchmark_report.md`; tuned results are written separately.

In [ ]:
!python3 scripts/machine_learning/tune_models.py --top-k 3
import json
t = json.load(open('results/machine_learning/benchmark/tuned_benchmark.json'))
cmp = pd.DataFrame(t['comparison_to_baseline'])
print('=== Baseline → tuned (same test split; search used train CV only) ===')
display(cmp.style.format({
    'baseline_cv_roc_auc':'{:.4f}','tuned_cv_roc_auc':'{:.4f}',
    'baseline_test_roc_auc':'{:.4f}','tuned_test_roc_auc':'{:.4f}',
    'baseline_test_f1':'{:.4f}','tuned_test_f1':'{:.4f}',
}).hide(axis='index'))

## 12 · Position-aware validation (separate experiment)
GroupKFold by residue — all substitutions at the same position stay in one fold (detects same-residue leakage vs the random split).

In [ ]:
!python3 scripts/machine_learning/position_aware_validation.py
pa = pd.read_csv('results/machine_learning/position_aware/position_aware_results.csv')
pa_out = pa[['model','cv_roc_auc_mean','cv_roc_auc_std','cv_f1_mean','cv_f1_std','cv_mcc_mean']].copy()
pa_out = pa_out.sort_values('cv_roc_auc_mean', ascending=False)
print('=== GroupKFold-by-position (compare to random-split CV in Table B) ===')
display(pa_out.style.format({
    'cv_roc_auc_mean':'{:.3f}','cv_roc_auc_std':'{:.3f}',
    'cv_f1_mean':'{:.3f}','cv_f1_std':'{:.3f}','cv_mcc_mean':'{:.3f}',
}).hide(axis='index'))

## 13 · Real mutant proteins (WT-validated single substitutions)

In [ ]:
!python3 scripts/mutation/build_mutant_sequences.py
mut = pd.read_csv('data/processed/mutations/mutant_sequences.csv')
print('lengths unique:', mut.length.unique(), '| sequences unique:', mut.md5.nunique() == len(mut))
!python3 scripts/mutation/write_variant_protein_results.py

## 14 · Full-cohort audit — every variant predicted, mistakes listed

In [ ]:
!python3 scripts/machine_learning/audit_predictions.py
pred = pd.read_csv('results/machine_learning/all_predictions.csv')
mis = pred[~pred.correct].sort_values('proba_pathogenic')
print(f"misclassified: {len(mis)} of {len(pred)} (test acc {(pred[pred.split=='test'].correct.mean()):.4f})")
display(mis[['hgvs_p','classification','split','proba_pathogenic']].reset_index(drop=True).head(25))
from IPython.display import Image
for f in ['figures/machine_learning/position_enrichment.png', 'figures/machine_learning/plddt_by_class.png',
          'figures/machine_learning/substitution_counts.png']:
    display(Image(f, width=660))

## 15 · Prediction panel — hotspot results

Default model = baseline RF (`model.joblib`). Optionally pass another benchmark artefact via `model_path=`.

In [ ]:
import sys; sys.path.insert(0, 'scripts/machine_learning')
from predict import predict_variant
rows = []
for v in [(175,'R','H'),(248,'R','Q'),(248,'R','W'),(273,'R','H'),(220,'Y','C'),(245,'G','S'),(72,'P','R')]:
    r = predict_variant(*v); c = r['context']
    rows.append({'variant': r['input']['hgvs_p'], 'prediction': r['result'],
                 'P(pathogenic)': r['probability_pathogenic'], 'domain': c['domain'],
                 'pLDDT': c['plddt_at_pos'], 'Grantham': c['grantham']})
display(pd.DataFrame(rows).set_index('variant'))

# Same variant through a different saved model (side-by-side inference)
print('\nR175H across saved models:')
import joblib, os
inf_rows = []
for path in sorted(os.listdir('results/machine_learning/benchmark/models')):
    if not path.endswith('.joblib'): continue
    p = os.path.join('results/machine_learning/benchmark/models', path)
    r = predict_variant(175,'R','H', model_path=p)
    inf_rows.append({'model_artefact': path, 'pred': r['result'],
                     'P(pathogenic)': r['probability_pathogenic']})
display(pd.DataFrame(inf_rows))

## 16 · WT structure with hotspot positions (static 3D, real model_4 coordinates)

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
tr = json.load(open('dashboard/ca_trace.json'))['residues']
xs = [r['x'] for r in tr]; ys = [r['y'] for r in tr]; zs = [r['z'] for r in tr]; pl = [r['plddt'] for r in tr]
HOT = {175:'R175H', 248:'R248Q', 273:'R273H', 220:'Y220C', 72:'P72R'}
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(xs, ys, zs, color='#9aa3b2', lw=1, alpha=0.7)
sc = ax.scatter(xs, ys, zs, c=pl, cmap='viridis', s=6, vmin=0, vmax=100)
for pos, lab in HOT.items():
    r = tr[pos-1]
    ax.scatter([r['x']],[r['y']],[r['z']], color='#b4232a', s=60)
    ax.text(r['x'], r['y'], r['z'], lab, color='#b4232a', fontsize=9)
fig.colorbar(sc, ax=ax, shrink=0.6, label='pLDDT')
ax.set_title('WT TP53 CA trace (AlphaFold model_4) — red = demo variant positions')
ax.set_axis_off()
fig.tight_layout(); fig.savefig('figures/structures/kaggle_3d_hotspots.png', dpi=150)
from IPython.display import Image
display(Image('figures/structures/kaggle_3d_hotspots.png', width=680))

## 17 · Smoke tests

In [ ]:
!python3 scripts/machine_learning/test_benchmark_smoke.py

## 18 · Done — what this notebook proves

WT verified (100% P04637) · **~818 ClinVar missense variants** (clear labels + conflicting majority-vote expansion, provenance in `label_basis`) · matching mutant proteins · 21 leak-free features · **3 classifiers under one fair protocol** with side-by-side test and CV tables + comparison figures · train-only hyperparameter search kept separate · position-aware GroupKFold as a leakage check · full-cohort audit.

Scores are **research-prototype associations with curated ClinVar labels on a random stratified split — not clinical performance**. Same-residue leakage across random splits is a known limitation; see `position_aware/`.

The HTML dashboard (`dashboard/`) is a local-web demo of the baseline model and cannot be served from Kaggle.